In [ ]:
import jax_sn as jsn
%load_ext autoreload
%autoreload 2
import jax_sbgeom

import jax.numpy as jnp
import numpy as onp
from typing import Tuple
import jax
import numpy as onp
from functools import partial
jax.config.update("jax_enable_x64", True)

import pyvista as pv

def remove_zeros(array):
    a = onp.array(array)
    a[onp.abs(a) < 1e-14] = 0.0
    return a

from jax_sn.domain.geometric_domain import GeometricDomain, TopologicalSortedAngle
from jax_sn.solution_domain.solution_domain import FEMMatrices, PrecomputedGeometricData
from jax_sn.domain.geometric_domain import CrossSectionData, CrossSectionDataSingleGroup
from jax_sn.source import Source
from typing import List
from jax_sn.tools.mesh_tools import _pad_last_dimension
import matplotlib.pyplot as plt
from dataclasses import dataclass
onp.set_printoptions(linewidth = 200)

In [ ]:
from jax_sn.domain.domain_generation import create_cartesian_setup
from jax_sn.solution_domain import BasixLagrangianSimplex
from jax_sn.solution_domain import SolutionDomain
from jax_sn.domain import CrossSectionData, Domain

In [ ]:
from jax_sn.solution_domain import compute_setup_for_solution_domain, transport_inverse_sweep_complete_vmap
from jax_sn.solution_domain.transport_operator import transport_inverse_sweep_multiple_angles_batched, transport_inverse_sweep_multiple_angles_batched_single_f


sigma  = 5
xs_0   = CrossSectionData.constant(sigma, 1, 1)
q      = 5
dim    = 3
degree = 1
n_elem = 100
verts, conn, cross_sections, sources = create_cartesian_setup(dim, [[(1.0, n_elem)], [(0.2, 10)], [(0.2, 10)]], cross_sections_regions=[xs_0], source_regions=[q], inhomogeneity_direction=0)
element_type                         = BasixLagrangianSimplex(degree, dim)
domain                               = Domain.from_mesh_and_cross_sections((verts, conn), element_type.face_template, cross_sections_indexed=cross_sections)
solution_domain                      = SolutionDomain.from_element_and_domain(element_type, domain)
source_mapped                        = Source(jnp.repeat(sources, element_type.n_basis, axis=-1).reshape(-1, element_type.n_basis))
buffer_sweep                         = jnp.zeros((domain.n_elements, element_type.n_basis))
quad_set = jsn.quadrature_set.create_tn_quadrature_set(4)
toposorts = [TopologicalSortedAngle.from_domain_angle(domain, angle, 100) for angle in quad_set.angles]
total_toposorts = jax.tree.map(lambda * xs : jnp.stack(xs), *toposorts)
source_total = jax.tree.map(lambda * xs : jnp.stack(xs), *[source_mapped for _ in toposorts])